# 07. Classes and OOP

## What you'll learn

- How to define classes with `__init__`, `self`, and instance attributes
- Constructor patterns for flexible object creation
- Methods that read and modify object state
- Dunder (magic) methods: `__str__`, `__repr__`, `__len__`
- Inheritance with `super()` and method overriding
- When to use composition vs inheritance
- How to build a mini agent framework with tools and dispatch

## Prerequisites

- [01 — Python Fundamentals](01_python_fundamentals.ipynb): variables, basic types, control flow
- [02 — Functions and Scope](02_functions_and_scope.ipynb): defining functions, arguments, return values
- [03 — Data Structures](03_data_structures.ipynb): lists, dicts, iteration
- [04 — Strings and JSON](04_strings_and_json.ipynb): string manipulation, JSON parsing
- [05 — Error Handling](05_error_handling.ipynb): try/except, raising exceptions
- [06 — HTTP and APIs](06_http_and_apis.ipynb): making API calls (helpful context, not strictly required)

## Why this matters for agents

Every serious agent framework — smolagents, LangChain, CrewAI — is built on classes.
An agent is an object with state (conversation history, tools, configuration).
A tool is an object with a name, description, and a `run()` method.
Understanding OOP is the bridge between writing ad-hoc scripts and building real,
composable agent systems. The capstone in this notebook builds a mini agent framework
that mirrors the architecture of smolagents.

> **Further reading:**
> - [Python Classes tutorial](https://docs.python.org/3/tutorial/classes.html) — official Python docs on classes and OOP
> - [smolagents source code](https://github.com/huggingface/smolagents) — see how `Tool` and `Agent` classes are structured in a real framework

---

## 1. Class Basics: `class`, `__init__`, and `self`

A **class** is a blueprint for creating objects. Each object (instance) holds its own
data (attributes) and has access to shared behavior (methods).

The `__init__` method is the **constructor** — it runs when you create a new instance.
The `self` parameter is a reference to the instance being created or acted upon.

Let's start with something agent-relevant: a class to represent an LLM message.

In [ ]:
class Message:
    """Represents a single message in an LLM conversation."""

    def __init__(self, role, content):
        self.role = role        # "user", "assistant", or "system"
        self.content = content  # the text of the message


# Create instances
system_msg = Message("system", "You are a helpful assistant.")
user_msg = Message("user", "What is an AI agent?")

print(f"Role: {system_msg.role}")
print(f"Content: {system_msg.content}")
print(f"User says: {user_msg.content}")

Each `Message` instance has its own `role` and `content`. The `self.role = role` line
stores the argument as an **instance attribute** — data that belongs to that specific object.

Notice: `system_msg.role` and `user_msg.role` are independent. Changing one does not
affect the other.

In [ ]:
# Each instance is independent
system_msg.role = "system_prompt"  # modify one instance
print(f"system_msg.role: {system_msg.role}")
print(f"user_msg.role:   {user_msg.role}")  # unchanged

---

## 2. Constructor Patterns

Real-world classes often need flexible constructors: default values, validation,
and derived attributes. Let's build a `ToolDefinition` class that mirrors how
tools are defined in agent frameworks.

In [ ]:
class ToolDefinition:
    """Describes a tool that an agent can use."""

    def __init__(self, name, description, parameters=None):
        self.name = name
        self.description = description
        self.parameters = parameters or []  # default to empty list
        self.call_count = 0                  # derived attribute, always starts at 0


# With all arguments
search_tool = ToolDefinition(
    name="web_search",
    description="Search the web for information",
    parameters=["query"]
)

# With default parameters
clock_tool = ToolDefinition(
    name="get_time",
    description="Return the current time"
)

print(f"{search_tool.name}: params={search_tool.parameters}")
print(f"{clock_tool.name}: params={clock_tool.parameters}")

Key patterns to note:

- **Default arguments** (`parameters=None`) let callers omit optional fields.
- **`or []`** avoids the mutable default argument pitfall (never use `def f(x=[])`!).
- **Derived attributes** (`call_count`) are computed in `__init__`, not passed in.

You can also add **validation** in the constructor:

In [ ]:
class ValidatedToolDefinition:
    """ToolDefinition with constructor validation."""

    VALID_ROLES = {"search", "calculate", "transform", "retrieve"}

    def __init__(self, name, role, description):
        if not name:
            raise ValueError("Tool name cannot be empty")
        if role not in self.VALID_ROLES:
            raise ValueError(f"Invalid role '{role}'. Must be one of {self.VALID_ROLES}")
        self.name = name
        self.role = role
        self.description = description


# Valid construction
tool = ValidatedToolDefinition("calculator", "calculate", "Does math")
print(f"Created tool: {tool.name} ({tool.role})")

# Invalid construction — will raise an error
try:
    bad_tool = ValidatedToolDefinition("mystery", "magic", "Does magic")
except ValueError as e:
    print(f"Caught error: {e}")

---

## 3. Methods That Read and Modify State

Methods are functions defined inside a class. They always receive `self` as the first
argument, giving them access to the instance's attributes.

Let's build a `Conversation` class — the kind of thing every agent needs to manage
its message history.

In [ ]:
class Conversation:
    """Manages a list of messages for an LLM conversation."""

    def __init__(self, system_prompt="You are a helpful assistant."):
        self.messages = [
            {"role": "system", "content": system_prompt}
        ]

    def add_user_message(self, content):
        """Append a user message to the conversation."""
        self.messages.append({"role": "user", "content": content})

    def add_assistant_message(self, content):
        """Append an assistant response to the conversation."""
        self.messages.append({"role": "assistant", "content": content})

    def get_last_message(self):
        """Return the most recent message."""
        return self.messages[-1]

    def message_count(self):
        """Return the number of messages (excluding system prompt)."""
        return len(self.messages) - 1

    def to_api_format(self):
        """Return messages as a list of dicts (ready for an LLM API call)."""
        return list(self.messages)  # return a copy


# Use it
convo = Conversation("You are an agent that can search the web.")
convo.add_user_message("Find information about smolagents.")
convo.add_assistant_message("I'll search for that. Let me use my web_search tool.")

print(f"Messages so far: {convo.message_count()}")
print(f"Last message: {convo.get_last_message()}")
print(f"\nAPI format:")
for msg in convo.to_api_format():
    print(f"  [{msg['role']}] {msg['content'][:50]}")

Notice the pattern:

- **Read methods** (`get_last_message`, `message_count`, `to_api_format`) inspect state without changing it.
- **Mutating methods** (`add_user_message`, `add_assistant_message`) modify `self.messages`.

This separation makes the class predictable — you know which methods change state and which don't.

---

## 4. Dunder Methods: `__str__`, `__repr__`, `__len__`

Python uses **dunder (double-underscore) methods** to let your objects integrate
with built-in operations like `print()`, `len()`, and `repr()`.

| Dunder | Triggered by | Purpose |
|--------|-------------|----------|
| `__str__` | `print(obj)`, `str(obj)` | Human-readable representation |
| `__repr__` | `repr(obj)`, interpreter display | Unambiguous developer representation |
| `__len__` | `len(obj)` | Number of items in the object |

Let's enhance our `Conversation` class:

In [ ]:
class Conversation:
    """Manages a list of messages with nice display support."""

    def __init__(self, system_prompt="You are a helpful assistant."):
        self.system_prompt = system_prompt
        self.messages = [
            {"role": "system", "content": system_prompt}
        ]

    def add(self, role, content):
        self.messages.append({"role": role, "content": content})

    def __str__(self):
        """Human-readable summary."""
        lines = []
        for msg in self.messages:
            preview = msg["content"][:60]
            lines.append(f"  [{msg['role']:>9}] {preview}")
        return f"Conversation ({len(self)} messages):\n" + "\n".join(lines)

    def __repr__(self):
        """Unambiguous developer representation."""
        return f"Conversation(system_prompt={self.system_prompt!r}, messages={len(self.messages)})"

    def __len__(self):
        """Number of messages including system prompt."""
        return len(self.messages)


convo = Conversation("You are a coding assistant.")
convo.add("user", "How do I build an agent in Python?")
convo.add("assistant", "Start by defining a loop: prompt -> LLM -> parse -> act.")

# __str__ is used by print()
print(convo)
print()

# __repr__ is used by repr() and the REPL
print(f"repr: {convo!r}")

# __len__ is used by len()
print(f"len:  {len(convo)}")

A useful rule of thumb:

- `__str__`: What would a *user* want to see? Keep it readable.
- `__repr__`: What would a *developer debugging* want to see? Include the class name and key state.
- `__len__`: Only implement if the object is conceptually a *container* with a size.

---

## 5. Inheritance, `super()`, and Method Overriding

**Inheritance** lets you create a new class based on an existing one. The child class
inherits all methods and attributes from the parent, and can override or extend them.

This is the core pattern behind tool systems in agent frameworks: define a base
`Tool` class, then create specific tools as subclasses.

```
Tool (base)
 ├── SearchTool
 ├── CalculatorTool
 └── WeatherTool
```

In [ ]:
class Tool:
    """Base class for all agent tools."""

    name = "base_tool"
    description = "A generic tool"

    def __init__(self):
        self.call_count = 0

    def run(self, **kwargs):
        """Execute the tool. Subclasses MUST override this."""
        raise NotImplementedError("Subclasses must implement run()")

    def __str__(self):
        return f"Tool({self.name}: {self.description})"


# Try calling the base class directly
base = Tool()
print(base)
try:
    base.run()
except NotImplementedError as e:
    print(f"Error: {e}")

Now let's create concrete tool subclasses that override `run()`:

In [ ]:
class SearchTool(Tool):
    """Simulates a web search."""

    name = "web_search"
    description = "Search the web for information"

    def run(self, query):
        self.call_count += 1
        # In a real agent, this would call a search API
        return f"[Search results for '{query}': 3 results found]"


class CalculatorTool(Tool):
    """Evaluates simple math expressions safely."""

    name = "calculator"
    description = "Evaluate a mathematical expression"

    ALLOWED_CHARS = set("0123456789+-*/.() ")

    def run(self, expression):
        self.call_count += 1
        # Basic safety check
        if not all(c in self.ALLOWED_CHARS for c in expression):
            return "Error: expression contains invalid characters"
        try:
            result = eval(expression)  # safe here due to char whitelist
            return f"{expression} = {result}"
        except Exception as e:
            return f"Error: {e}"


# Use the subclasses
search = SearchTool()
calc = CalculatorTool()

print(search)  # __str__ inherited from Tool
print(search.run(query="smolagents tutorial"))
print()
print(calc)
print(calc.run(expression="(42 + 8) * 2"))
print(f"\nSearch called {search.call_count} time(s)")
print(f"Calc called {calc.call_count} time(s)")

Key inheritance concepts:

- `SearchTool(Tool)` means `SearchTool` **inherits from** `Tool`.
- `SearchTool` overrides `name`, `description`, and `run()` but **inherits** `__init__` and `__str__`.
- `isinstance(search, Tool)` is `True` — a `SearchTool` *is a* `Tool`.

Now let's see `super()` in action — calling the parent class's methods:

In [ ]:
class LoggingTool(Tool):
    """A tool that logs every call."""

    name = "logging_demo"
    description = "Demonstrates super() with logging"

    def __init__(self):
        super().__init__()       # call Tool.__init__() to set up call_count
        self.log = []            # add our own attribute

    def run(self, message):
        self.call_count += 1     # inherited attribute
        self.log.append(message) # our attribute
        return f"Logged: {message}"


logger = LoggingTool()
logger.run(message="first call")
logger.run(message="second call")

print(f"Call count: {logger.call_count}")
print(f"Log: {logger.log}")
print(f"Is a Tool? {isinstance(logger, Tool)}")

`super().__init__()` calls the parent's constructor, ensuring `call_count` is set up properly.
Without it, we'd get an `AttributeError` when trying to use `self.call_count`.

**Rule of thumb:** If your subclass defines `__init__`, always call `super().__init__()` first
(unless you have a very specific reason not to).

---

## 6. Composition vs Inheritance

Inheritance models an **"is-a"** relationship: a `SearchTool` *is a* `Tool`.

**Composition** models a **"has-a"** relationship: an `Agent` *has* tools, *has* a conversation.

In practice, agent frameworks use both:
- **Inheritance** for the tool hierarchy (all tools share a common interface).
- **Composition** for the agent itself (an agent *contains* tools and a conversation).

```
Agent
 ├── has-a Conversation
 ├── has-a list[Tool]
 │    ├── SearchTool (is-a Tool)
 │    └── CalculatorTool (is-a Tool)
 └── has-a config dict
```

In [ ]:
class AgentConfig:
    """Configuration object for an agent."""

    def __init__(self, model="gpt-3.5-turbo", max_steps=5, verbose=False):
        self.model = model
        self.max_steps = max_steps
        self.verbose = verbose

    def __repr__(self):
        return f"AgentConfig(model={self.model!r}, max_steps={self.max_steps})"


class SimpleAgent:
    """An agent that COMPOSES a config, conversation, and tools."""

    def __init__(self, config, tools=None):
        self.config = config               # has-a AgentConfig
        self.conversation = Conversation()  # has-a Conversation
        self.tools = tools or []            # has-a list of Tools

    def available_tools(self):
        """List the names of all registered tools."""
        return [tool.name for tool in self.tools]

    def __str__(self):
        tool_names = ", ".join(self.available_tools()) or "none"
        return f"SimpleAgent(model={self.config.model}, tools=[{tool_names}])"


# Compose an agent from independent parts
config = AgentConfig(model="mistral-7b", max_steps=10)
agent = SimpleAgent(
    config=config,
    tools=[SearchTool(), CalculatorTool()]
)

print(agent)
print(f"Config: {agent.config!r}")
print(f"Conversation length: {len(agent.conversation)}")
print(f"Tools: {agent.available_tools()}")

Why composition over inheritance here?

- An `Agent` is not a kind of `Tool` or `Conversation` — it *uses* them.
- You can swap out `AgentConfig` or `Conversation` independently.
- You can test each component in isolation.

**Guideline:** Prefer composition when objects *work together*. Use inheritance when objects
share a *common interface* (like all tools having a `run()` method).

---

## Putting It Together: A Mini Agent Framework

Let's combine everything into a small but functional agent framework. This mirrors
the actual architecture of smolagents:

1. A `Tool` base class with `name`, `description`, and `run()`
2. Concrete tool subclasses (`SearchTool`, `CalculatorTool`)
3. An `Agent` class that holds tools and a conversation, with a `dispatch()` method
   to route tool calls by name

This is the pattern you'll see everywhere in agent development.

In [ ]:
class Tool:
    """Base class for all agent tools (framework version)."""

    name = "base_tool"
    description = "Override this in subclasses"

    def __init__(self):
        self.call_count = 0

    def run(self, **kwargs):
        raise NotImplementedError("Subclasses must implement run()")

    def __repr__(self):
        return f"{self.__class__.__name__}(name={self.name!r})"

    def __str__(self):
        return f"{self.name}: {self.description}"

In [ ]:
class SearchTool(Tool):
    """Simulates searching the web."""

    name = "web_search"
    description = "Search the web and return results"

    # Simulated database of knowledge
    KNOWLEDGE = {
        "smolagents": "smolagents is a lightweight agent framework by HuggingFace.",
        "python": "Python is a versatile programming language created by Guido van Rossum.",
        "oop": "Object-Oriented Programming organizes code around objects and classes.",
    }

    def run(self, query):
        self.call_count += 1
        query_lower = query.lower()
        for key, value in self.KNOWLEDGE.items():
            if key in query_lower:
                return value
        return f"No results found for '{query}'."


class CalculatorTool(Tool):
    """Evaluates simple math expressions."""

    name = "calculator"
    description = "Evaluate a mathematical expression"

    ALLOWED_CHARS = set("0123456789+-*/.() ")

    def run(self, expression):
        self.call_count += 1
        if not all(c in self.ALLOWED_CHARS for c in expression):
            return "Error: invalid characters in expression"
        try:
            return str(eval(expression))
        except Exception as e:
            return f"Error: {e}"

In [ ]:
class Agent:
    """A minimal agent that can dispatch tool calls."""

    def __init__(self, tools, system_prompt="You are a helpful agent."):
        # Build a name -> tool lookup dict (composition)
        self.tool_registry = {}
        for tool in tools:
            self.tool_registry[tool.name] = tool

        self.history = [
            {"role": "system", "content": system_prompt}
        ]

    def dispatch(self, tool_name, **kwargs):
        """Look up a tool by name and execute it."""
        if tool_name not in self.tool_registry:
            available = list(self.tool_registry.keys())
            return f"Error: unknown tool '{tool_name}'. Available: {available}"

        tool = self.tool_registry[tool_name]
        result = tool.run(**kwargs)

        # Log the tool call in history
        self.history.append({
            "role": "assistant",
            "content": f"[Called {tool_name} with {kwargs}] -> {result}"
        })

        return result

    def list_tools(self):
        """Return a formatted list of available tools."""
        lines = []
        for name, tool in self.tool_registry.items():
            lines.append(f"  - {tool}")
        return "\n".join(lines)

    def get_stats(self):
        """Return usage statistics for all tools."""
        return {
            name: tool.call_count
            for name, tool in self.tool_registry.items()
        }

    def __repr__(self):
        tool_names = list(self.tool_registry.keys())
        return f"Agent(tools={tool_names}, history_len={len(self.history)})"

    def __len__(self):
        """Number of entries in history."""
        return len(self.history)

In [ ]:
# --- Build and use the agent ---

agent = Agent(
    tools=[SearchTool(), CalculatorTool()],
    system_prompt="You are a research assistant with access to tools."
)

print("Available tools:")
print(agent.list_tools())
print()

# Dispatch some tool calls (simulating what an LLM would request)
result1 = agent.dispatch("web_search", query="What is smolagents?")
print(f"Search result: {result1}")

result2 = agent.dispatch("calculator", expression="256 * 3 + 42")
print(f"Calc result:   {result2}")

result3 = agent.dispatch("unknown_tool", query="test")
print(f"Unknown tool:  {result3}")

print(f"\nAgent state: {agent!r}")
print(f"Tool usage stats: {agent.get_stats()}")

print(f"\nHistory ({len(agent)} entries):")
for entry in agent.history:
    print(f"  [{entry['role']}] {entry['content'][:70]}")

This mini framework demonstrates all the OOP concepts from this notebook:

| Concept | Where it appears |
|---------|------------------|
| `__init__` / `self` | Every class |
| Constructor patterns | `Agent.__init__` builds the tool registry from a list |
| Read/modify methods | `dispatch()` modifies history; `list_tools()` reads state |
| Dunder methods | `__repr__`, `__str__`, `__len__` on `Agent` and `Tool` |
| Inheritance | `SearchTool(Tool)`, `CalculatorTool(Tool)` |
| Composition | `Agent` *has-a* dict of `Tool` objects and a history list |

When you move to smolagents in the core track, you'll see the exact same architecture:
a `Tool` base class with a `forward()` method, and an `Agent` that manages tools,
conversation history, and an LLM.

---

## Try It Yourself

### Exercise 1: Counter Class

Create a `Counter` class with:
- An `__init__` that takes an optional `start` value (default 0)
- An `increment()` method (adds 1)
- A `decrement()` method (subtracts 1, but never goes below 0)
- A `reset()` method (resets to the start value)
- `__str__` that returns `"Counter(value=N)"`
- `__repr__` that returns `"Counter(start=S, value=N)"`

In [ ]:
# Exercise 1: Your code here

# class Counter:
#     ...

# Test it:
# c = Counter(start=5)
# c.increment()
# c.increment()
# print(c)           # Counter(value=7)
# c.decrement()
# print(c)           # Counter(value=6)
# c.reset()
# print(repr(c))     # Counter(start=5, value=5)

### Exercise 2: Vehicle Hierarchy

Build a small inheritance hierarchy:

1. A `Vehicle` base class with `__init__(self, name, max_speed)` and a `describe()` method
2. A `Car` subclass that adds a `num_doors` attribute and overrides `describe()`
3. A `Bicycle` subclass that adds a `gear_count` attribute and overrides `describe()`

Use `super().__init__()` in the subclass constructors.

In [ ]:
# Exercise 2: Your code here

# class Vehicle:
#     ...

# class Car(Vehicle):
#     ...

# class Bicycle(Vehicle):
#     ...

# Test it:
# car = Car("Tesla Model 3", max_speed=150, num_doors=4)
# bike = Bicycle("Trek FX", max_speed=30, gear_count=21)
# print(car.describe())
# print(bike.describe())
# print(isinstance(car, Vehicle))   # True
# print(isinstance(bike, Vehicle))  # True

### Exercise 3: MemoryAgent with Conversation History

Create a `MemoryAgent` that inherits from the `Agent` class defined in the capstone.
Add these capabilities:

1. A `chat(user_input)` method that:
   - Adds the user input to history
   - Generates a simple response (e.g., echo it back or use a template)
   - Adds the response to history
   - Returns the response

2. A `summarize_history()` method that returns a summary string like:
   `"3 turns: user asked about X, Y, Z"`

3. A `clear_history()` method that resets history but keeps the system prompt.

In [ ]:
# Exercise 3: Your code here

# class MemoryAgent(Agent):
#     ...

# Test it:
# mem_agent = MemoryAgent(tools=[SearchTool()])
# mem_agent.chat("Tell me about Python")
# mem_agent.chat("What about OOP?")
# print(mem_agent.summarize_history())
# print(f"History length: {len(mem_agent)}")
# mem_agent.clear_history()
# print(f"After clear: {len(mem_agent)}")

### Exercise 4 (Stretch): Add a New Tool

Create a `FormatterTool` subclass of `Tool` that takes `text` and `style` parameters.
It should transform the text based on the style:

- `"upper"` -> uppercase
- `"lower"` -> lowercase
- `"title"` -> title case
- anything else -> return the text unchanged

Register it with an `Agent` and dispatch a call to it.

In [ ]:
# Exercise 4: Your code here

# class FormatterTool(Tool):
#     ...

# Test it:
# agent = Agent(tools=[FormatterTool(), SearchTool()])
# print(agent.dispatch("formatter", text="hello world", style="upper"))
# print(agent.dispatch("formatter", text="HELLO WORLD", style="title"))